In [2]:
import yfinance
from kronos import Kronos, KronosTokenizer, KronosPredictor
import plotly.graph_objects as go

# 0. Load SPY data

In [4]:
spy = yfinance.Ticker("SPY")
spy = spy.history(period="max").reset_index()
spy

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Capital Gains
0,1993-01-29 00:00:00-05:00,24.192590,24.192590,24.072229,24.175396,1003200,0.0,0.0,0.0
1,1993-02-01 00:00:00-05:00,24.192580,24.347330,24.192580,24.347330,480500,0.0,0.0,0.0
2,1993-02-02 00:00:00-05:00,24.330129,24.416101,24.278546,24.398907,201300,0.0,0.0,0.0
3,1993-02-03 00:00:00-05:00,24.433308,24.674030,24.416113,24.656836,529400,0.0,0.0,0.0
4,1993-02-04 00:00:00-05:00,24.742792,24.811570,24.467681,24.759987,531500,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
8369,2026-04-30 00:00:00-04:00,714.630005,719.789978,710.450012,718.659973,67240900,0.0,0.0,0.0
8370,2026-05-01 00:00:00-04:00,721.250000,724.869995,720.469971,720.650024,43049800,0.0,0.0,0.0
8371,2026-05-04 00:00:00-04:00,720.070007,722.119995,714.989990,718.010010,51950600,0.0,0.0,0.0
8372,2026-05-05 00:00:00-04:00,721.770020,725.039978,721.489990,723.770020,36933200,0.0,0.0,0.0


# 1. Load model and tokenizer

In [ ]:
tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
model = Kronos.from_pretrained("NeoQuasar/Kronos-base")

# 2. Instantiate Predictor

In [6]:
predictor = KronosPredictor(model, tokenizer, max_context=512)

# 3. Prepare data

In [7]:
lookback = 400
pred_len = 120

In [10]:
x_df = spy.loc[:lookback-1, ['Open', 'High', 'Low', 'Close', 'Volume']].rename(
    columns={
        'Open': 'open',
        'High': 'high',
        'Low': 'low',
        'Close': 'close',
        'Volume': 'volume'
    }
)
x_timestamp = spy.loc[:lookback-1, 'Date']
y_timestamp = spy.loc[lookback:lookback+pred_len-1, 'Date']

In [14]:
x_timestamp

0     1993-01-29 00:00:00-05:00
1     1993-02-01 00:00:00-05:00
2     1993-02-02 00:00:00-05:00
3     1993-02-03 00:00:00-05:00
4     1993-02-04 00:00:00-05:00
                 ...           
395   1994-08-23 00:00:00-04:00
396   1994-08-24 00:00:00-04:00
397   1994-08-25 00:00:00-04:00
398   1994-08-26 00:00:00-04:00
399   1994-08-29 00:00:00-04:00
Name: Date, Length: 400, dtype: datetime64[ns, America/New_York]

# 4. Make Prediction

In [18]:
pred_df = predictor.predict(
    df=x_df,
    x_timestamp=x_timestamp,
    y_timestamp=y_timestamp,
    pred_len=pred_len,
    T=1.0,
    top_p=0.9,
    sample_count=1,
    verbose=True
).reset_index()

  0%|          | 0/120 [00:00<?, ?it/s]

100%|██████████| 120/120 [00:23<00:00,  5.11it/s]


In [19]:
pred_df

,Date,open,high,low,close,volume,amount
0,1994-08-30 00:00:00-04:00,27.354774,27.499285,27.316387,27.458632,2.339479e+05,6216747.0
1,1994-08-31 00:00:00-04:00,27.442949,27.772949,26.458696,27.342037,1.873426e+06,46287480.0
2,1994-09-01 00:00:00-04:00,27.190113,27.255369,26.797888,26.845783,3.229562e+05,8308379.0
3,1994-09-02 00:00:00-04:00,26.919586,27.071598,26.916054,27.048944,1.023786e+05,2533496.0
4,1994-09-06 00:00:00-04:00,26.876654,26.890181,26.714491,26.733906,8.962256e+04,2112247.0
...,...,...,...,...,...,...,...
115,1995-02-13 00:00:00-05:00,25.987473,25.978092,25.925648,25.870304,7.994687e+05,20378656.0
116,1995-02-14 00:00:00-05:00,25.960045,26.485622,25.904770,25.944323,3.165714e+05,8621828.0
117,1995-02-15 00:00:00-05:00,25.932867,25.964216,25.880007,25.921141,3.358644e+05,8619996.0
118,1995-02-16 00:00:00-05:00,25.932547,25.938822,25.719395,25.749105,3.949218e+05,10347303.0


# 5. Visualize Results

In [16]:
kline_df = spy.loc[:lookback+pred_len-1]
kline_df

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Capital Gains
0,1993-01-29 00:00:00-05:00,24.192590,24.192590,24.072229,24.175396,1003200,0.0,0.0,0.0
1,1993-02-01 00:00:00-05:00,24.192580,24.347330,24.192580,24.347330,480500,0.0,0.0,0.0
2,1993-02-02 00:00:00-05:00,24.330129,24.416101,24.278546,24.398907,201300,0.0,0.0,0.0
3,1993-02-03 00:00:00-05:00,24.433308,24.674030,24.416113,24.656836,529400,0.0,0.0,0.0
4,1993-02-04 00:00:00-05:00,24.742792,24.811570,24.467681,24.759987,531500,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
515,1995-02-13 00:00:00-05:00,27.986847,28.068283,27.986847,28.013992,79700,0.0,0.0,0.0
516,1995-02-14 00:00:00-05:00,28.104483,28.104483,27.941610,28.050192,170200,0.0,0.0,0.0
517,1995-02-15 00:00:00-05:00,28.050179,28.240196,28.023033,28.204002,431500,0.0,0.0,0.0
518,1995-02-16 00:00:00-05:00,28.167828,28.194973,28.086392,28.167828,99300,0.0,0.0,0.0


In [22]:
go.Figure(
    data=[
        go.Candlestick(
            x=kline_df['Date'],
            open=kline_df['Open'],
            high=kline_df['High'],
            low=kline_df['Low'],
            close=kline_df['Close'],
            increasing_line_color='green',
            name="Ground Truth"
        ),
        go.Candlestick(
            x=pred_df['Date'],
            open=pred_df['open'],
            high=pred_df['high'],
            low=pred_df['low'],
            close=pred_df['close'],
            increasing_line_color='purple',
            decreasing_line_color='orange',
            name="Prediction"
        )
    ]
).show()